
# LLM Chatbot — Jupyter Version

A chatbot built on top of an LLM API (Anthropic Claude by default). This started as a plain script but I split it into cells so I can poke at individual pieces (test the trimming logic, swap personas, replay a saved session) without re-running the whole thing every time.

**What it handles that a "20-line chatbot" tutorial doesn't:**
- Token-aware history trimming so the context doesn't blow up over a long session
- Retries with backoff on rate limits / connection errors
- Save/load sessions to JSON
- A small command layer for persona switching, history dumps, resets

Run the setup cells top to bottom once, then use the chat cell at the bottom as many times as you want.

*Author: Parul*


## 1. Setup

In [ ]:
# pip install anthropic   (uncomment and run once if not installed)
# !pip install anthropic

import os
import json
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path

import anthropic


### API key

Set this as an environment variable before launching Jupyter (`export ANTHROPIC_API_KEY=...`)
rather than hardcoding it in a cell — don't want that ending up in a saved `.ipynb` or a git push.


In [ ]:
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
assert API_KEY, "ANTHROPIC_API_KEY not set — export it in your shell before starting Jupyter"


## 2. Config

In [ ]:
DEFAULT_MODEL = "claude-sonnet-4-6"
MAX_TOKENS_RESPONSE = 1024

# rough heuristic - 1 token ~ 4 chars in English. Not exact but good
# enough to decide when to start trimming history instead of calling
# a real tokenizer every single turn.
CHARS_PER_TOKEN = 4
MAX_CONTEXT_TOKENS = 6000  # keep headroom below the model's real limit

HISTORY_DIR = Path("chat_sessions")
HISTORY_DIR.mkdir(exist_ok=True)

DEFAULT_SYSTEM_PROMPT = (
    "You are a helpful, direct assistant. Keep answers concise unless "
    "the user asks for detail. If you don\'t know something, say so "
    "instead of guessing."
)


## 3. Data model — `Message` and `Session`

In [ ]:
@dataclass
class Message:
    role: str          # "user" or "assistant"
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    def to_api_format(self):
        # the API doesn\'t need our timestamp, just role + content
        return {"role": self.role, "content": self.content}


@dataclass
class Session:
    session_id: str
    system_prompt: str
    messages: list = field(default_factory=list)

    def add(self, role, content):
        self.messages.append(Message(role=role, content=content))

    def save(self):
        path = HISTORY_DIR / f"{self.session_id}.json"
        data = {
            "session_id": self.session_id,
            "system_prompt": self.system_prompt,
            "messages": [asdict(m) for m in self.messages],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        return path

    @classmethod
    def load(cls, session_id):
        path = HISTORY_DIR / f"{session_id}.json"
        if not path.exists():
            raise FileNotFoundError(f"No saved session called \'{session_id}\'")
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        s = cls(session_id=data["session_id"], system_prompt=data["system_prompt"])
        for m in data["messages"]:
            s.messages.append(Message(role=m["role"], content=m["content"], timestamp=m["timestamp"]))
        return s


## 4. Core chatbot class

In [ ]:
class LLMChatbot:
    def __init__(self, api_key=None, model=DEFAULT_MODEL, system_prompt=DEFAULT_SYSTEM_PROMPT,
                 session_id=None):
        api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError(
                "No API key found. Set ANTHROPIC_API_KEY as an env var or pass it directly."
            )

        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model
        session_id = session_id or datetime.now().strftime("session_%Y%m%d_%H%M%S")
        self.session = Session(session_id=session_id, system_prompt=system_prompt)

    # ---- context management -------------------------------------

    def _estimate_tokens(self, text):
        return max(1, len(text) // CHARS_PER_TOKEN)

    def _trim_history(self):
        """
        Drop oldest messages if the running total gets too big.
        System prompt is passed separately at call time, so it\'s
        never at risk of getting trimmed out.
        """
        total = sum(self._estimate_tokens(m.content) for m in self.session.messages)
        while total > MAX_CONTEXT_TOKENS and len(self.session.messages) > 2:
            removed = self.session.messages.pop(0)
            total -= self._estimate_tokens(removed.content)

    # ---- API call with retries ------------------------------------

    def _call_llm(self, max_retries=3):
        self._trim_history()
        api_messages = [m.to_api_format() for m in self.session.messages]

        delay = 1.5
        last_err = None
        for attempt in range(1, max_retries + 1):
            try:
                response = self.client.messages.create(
                    model=self.model,
                    max_tokens=MAX_TOKENS_RESPONSE,
                    system=self.session.system_prompt,
                    messages=api_messages,
                )
                return response.content[0].text
            except anthropic.RateLimitError as e:
                last_err = e
                print(f"  (rate limited, retrying in {delay:.1f}s...)")
                time.sleep(delay)
                delay *= 2
            except anthropic.APIConnectionError as e:
                last_err = e
                print(f"  (connection issue, retrying in {delay:.1f}s...)")
                time.sleep(delay)
                delay *= 2
            except anthropic.APIStatusError as e:
                # server errors are worth retrying, client errors (bad request) are not
                if 500 <= e.status_code < 600:
                    last_err = e
                    time.sleep(delay)
                    delay *= 2
                else:
                    raise

        raise RuntimeError(f"Gave up after {max_retries} attempts: {last_err}")

    # ---- public interface -------------------------------------------

    def send(self, user_input):
        self.session.add("user", user_input)
        try:
            reply = self._call_llm()
        except RuntimeError:
            # don\'t corrupt history with a failed turn
            self.session.messages.pop()
            raise
        self.session.add("assistant", reply)
        return reply

    def reset(self):
        self.session.messages = []

    def set_system_prompt(self, prompt):
        self.session.system_prompt = prompt

    def save_session(self):
        return self.session.save()

    @classmethod
    def resume(cls, session_id, api_key=None, model=DEFAULT_MODEL):
        bot = cls(api_key=api_key, model=model, session_id=session_id)
        bot.session = Session.load(session_id)
        return bot


## 5. Spin up a bot

In [ ]:
bot = LLMChatbot(api_key=API_KEY)
print(f"session id: {bot.session.session_id}")


## 6. Chat

Run this cell as many times as you want — it keeps appending to the same session,
so the bot remembers earlier turns until you call `bot.reset()`.


In [ ]:
user_input = "Explain the difference between a hash map and a hash set in one paragraph."
reply = bot.send(user_input)
print("you: ", user_input)
print("bot: ", reply)


In [ ]:
user_input = "Now give me a one-line analogy for it."
reply = bot.send(user_input)
print("you: ", user_input)
print("bot: ", reply)


## 7. Utility cells — persona, history, save/load

In [ ]:
# change persona mid-session
bot.set_system_prompt(
    "You are a blunt, no-nonsense coding mentor. Point out mistakes directly, "
    "don't sugarcoat, but always suggest the fix."
)


In [ ]:
# dump the full conversation log
for m in bot.session.messages:
    print(f"[{m.timestamp}] {m.role}: {m.content}\n")


In [ ]:
# clear history but keep the current persona
bot.reset()


In [ ]:
# save to chat_sessions/<session_id>.json
path = bot.save_session()
print(f"saved to {path}")


In [ ]:
# resume a previous session by id (swap in a real session_id from chat_sessions/)
# bot = LLMChatbot.resume("session_20260810_120000", api_key=API_KEY)


## 8. Optional — interactive loop

Works in Jupyter's `input()` prompt too, just less convenient than running chat cells
individually. Interrupt the kernel (stop button) to break out.


In [ ]:
def chat_loop(bot):
    print("Type \'exit\' to stop.\n")
    while True:
        user_input = input("you: ").strip()
        if not user_input:
            continue
        if user_input.lower() == "exit":
            bot.save_session()
            print("saved. bye.")
            break
        try:
            reply = bot.send(user_input)
            print(f"bot: {reply}\n")
        except RuntimeError as e:
            print(f"[error] {e}\n")

# chat_loop(bot)   # uncomment to run
